In [ ]:
import sys
import antipode
from antipode.antipode_model import *
import antipode.model_functions
from antipode.model_functions import *
import antipode.model_distributions
from antipode.model_distributions import *
import antipode.model_modules
from antipode.model_modules import *
import antipode.train_utils
from antipode.train_utils import *
import antipode.plotting
from antipode.plotting import *
import torch
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU is available")
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available")


In [ ]:
antipode_model=antipode.antipode_model.ANTIPODE.load('/home/matthew.schmitz/Matthew/models/1.9.1.8.1_PsiNormal-Relu-final/',prefix='p3_',device=device)#adata=adata
adata=antipode_model.adata_manager.adata

In [1]:

num_features = antipode_model.adata_manager.adata.obsm['X_antipode'].shape[1]
num_components = antipode_model.level_sizes[-1]
batch_size = 256

# Create dataset
dataset = TensorDataset(torch.tensor(adata.obsm['X_antipode']))
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

def model(data):
    pi = pyro.sample("pi", dist.Dirichlet(0.5 * torch.ones(num_components)))
    with pyro.plate("components", num_components):
        locs = pyro.sample("locs", dist.Normal(torch.zeros(num_features), 10 * torch.ones(num_features)))
        scales = pyro.sample("scales", dist.LogNormal(torch.zeros(num_features), torch.ones(num_features)))
    with pyro.plate("data", data.size(0)):
        assignment = pyro.sample("assignment", dist.Categorical(pi))
        pyro.sample("obs", dist.Normal(locs[assignment], scales[assignment]), obs=data)

def guide(data):
    kappa = pyro.param("kappa", torch.ones(num_components), constraint=dist.constraints.positive)
    mu = pyro.param("mu", torch.randn(num_components, num_features))
    sigma = pyro.param("sigma", torch.ones(num_components, num_features), constraint=dist.constraints.positive)
    pi = pyro.sample("pi", dist.Dirichlet(kappa))
    with pyro.plate("components", num_components):
        locs = pyro.sample("locs", dist.Normal(mu, sigma))
        scales = pyro.sample("scales", dist.LogNormal(torch.log(sigma), 0.1 * torch.ones(num_features)))

optimizer = Adam({"lr": 0.02})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

num_iterations = 5000
for epoch in range(num_iterations):
    loss = 0
    for batch in dataloader:
        batch = batch[0]  # DataLoader packs data in tuples
        loss += svi.step(batch)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss / len(dataloader.dataset)}")

NameError: name 'antipode_model' is not defined